In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch, json
from tqdm import tqdm

# ---- Paths ----
A_PATH = "/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/training_sets/train_20251024_024220.csv"  # queries
B_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/4500_Posts_Annotations - Combined_Dataset.csv"           # reference pool
A_EMB_PATH = "/home/ubuntu/embeddings/test_500_train_4500_embeddings_A.pt"
B_EMB_PATH = "/home/ubuntu/embeddings/test_500_train_4500_embeddings_B.pt"
SIM_JSON_OUT = "/home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_4500_train_500_test.json"

# ---- Read ----
A = pd.read_csv(A_PATH)
B = pd.read_csv(B_PATH)

# ---- Build full_text (same style as your notebook) ----
A['full_text'] = A['title'].fillna('') + '. ' + A['selftext'].fillna('')
B['full_text'] = B.get('title','').fillna('') + '. ' + B.get('body','').fillna('')  # robust to missing cols

A.head(), B.head()


(        id    subreddit                                              title  \
 0  1nreglr  miscarriage               Constant Dull Pain on the right side   
 1  1nx3za3  miscarriage                   Everyone else forgets so quickly   
 2  1l7iqe8     abortion             Just needed to say this somewhere safe   
 3   jdmwi8      assault  I was touched by my cousin. (Posted in other s...   
 4   gg1xvz        metoo                           Me Too, But Not Anymore.   
 
                                             selftext          created_utc  \
 0  Hi! I (30F) recently experienced a miscarriage...  2025-09-26T22:08:33   
 1  My nephew’s fiancée is in labor and I keep get...  2025-10-03T16:31:24   
 2  Not really asking anything just needed to say ...  2025-06-09 22:37:08   
 3  so i would occasionally go to my grandparents ...  2020-10-18 19:47:54   
 4  I was sexually assaulted last year, by a famil...  2020-05-08 21:20:12   
 
                                                  url 

In [3]:
model = SentenceTransformer("all-mpnet-base-v2")

# Encode and save A
test13_emb_A = model.encode(
    A['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(test13_emb_A, A_EMB_PATH)

# Encode and save B (do once; later you can just torch.load(B_EMB_PATH))
test13_emb_B = model.encode(
    B['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(test13_emb_B, B_EMB_PATH)


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/141 [00:00<?, ?it/s]

In [4]:
emb_A = torch.load(A_EMB_PATH)
emb_B = torch.load(B_EMB_PATH)

In [5]:
k_max = 3

# Cosine similarity matrix: [len(A), len(B)]
# (normalize_embeddings=True above ⇒ cosine == dot)
sim_mat = util.cos_sim(test13_emb_A, test13_emb_B)  # torch tensor

# Top-k along B axis for each A row
top_vals, top_idx = torch.topk(sim_mat, k=k_max, dim=1)  # shapes: [len(A), k]

# Pack to dict: {a_row_index: [(b_index, score), ...]}
similar_posts = {}
for i in range(top_idx.size(0)):
    indices = top_idx[i].tolist()
    scores  = top_vals[i].tolist()
    similar_posts[i] = list(zip(indices, scores))

# Save JSON
with open(SIM_JSON_OUT, "w") as f:
    json.dump(similar_posts, f)

print(f"Saved cross-sim results to {SIM_JSON_OUT}")

Saved cross-sim results to /home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_4500_train_500_test.json
